# dragon_soul_winrate
[← back to readme](../readme.md)

I am comparing my winrate on teams with at least 4 dragon kills versus teams with fewer than 4 dragon kills corresponding to obtaining the dragon soul.
My data is somewhat limited ~ I only have data saying how many dragon kills a team got, not exactly whether they obtained the dragon soul. I am hereby making the assumption that they are equivalent and that killing the elder dragon while having three dragon kills is rare enough s. t. it won't significantly impact the result.

I am using a permutation test with a 5 / 3 % significance threshold.
The 3 is a part of Bonferroni's correction since I am testing 3 separate hypotheses.

## Imports

In [1]:
import csv

from permutation_distribution import *

## Load the aggregated data

In [2]:
rows = []

with open("dragon_soul_data.csv", "r", encoding="utf-8") as f:
  reader = csv.reader(f)

  next(reader)  # skip header

  for win, dragon_kills in reader:
    rows.append((win == "True", int(dragon_kills)))

mixed_sample = [win for win, _ in rows]

## Compare win rates

In [3]:
soul_sample = [win for win, dragon_kills in rows if dragon_kills >= 4]
soul_winrate = sum(soul_sample) / len(soul_sample)
print(f"{soul_winrate=:.2f}")
print(f"{len(soul_sample)=}")

print()

soulless_sample = [win for win, dragon_kills in rows if dragon_kills < 4]
soulless_winrate = sum(soulless_sample) / len(soulless_sample)
print(f"{soulless_winrate=:.2f}")
print(f"{len(soulless_sample)=}")
print()

soul_winrate=0.79
len(soul_sample)=39

soulless_winrate=0.44
len(soulless_sample)=185



## Permutation test

In [5]:
def wr_difference(soul_sample, soulless_sample):
  soul_winrate = sum(soul_sample) / len(soul_sample)
  soulless_winrate = sum(soulless_sample) / len(soulless_sample)

  return soul_winrate - soulless_winrate


dist = permutation_distribution(
  mixed_sample,
  wr_difference,
  len(soulless_sample)
)

significant_val = significant_value(dist, top_percentile=100 - 5 / 3)
observed_diff = soul_winrate - soulless_winrate

print(f"wr difference needed to be significant: {significant_val:.2f}")
print(f"observed wr difference: {observed_diff:.2f}")

wr difference needed to be significant: 0.20
observed wr difference: 0.36


The observed win-rate gap is well above the significance threshold, so the null hypothesis can be rejected.

## Conclusion
This result is to be expected. A team obtaining the dragon soul usually means the team is far ahead already and the soul is just the last nail in the coffin, not necessarily the sheer cause.

A further study might include plotting out the wr differences grouped by dragon counts and seeing if the dragon soul marks a significant jump compared to the other dragon kills. It would also be interesting to try controlling for gold lead, kill lead and other significant factors influencing the winrate. That is however not a part of the project specification and I will thus not do them.